# Speech Recognition — Colab Training

Train the LSTM-CTC model on **Google Colab GPU**. Your computer can be off — training runs on Google's servers.

**Before running:** Runtime → Change runtime type → **T4 GPU**

**Steps:**
1. Clone repo
2. Install dependencies
3. Download LibriSpeech dataset (~6 GB, one-time per session)
4. Train on full dataset
5. Save model to Google Drive (optional but recommended)

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Enable Runtime → Change runtime type → T4 GPU')

In [ ]:
# Clone the project (replace with your fork URL if needed)
!git clone https://github.com/sumathis15/Speech_Recognition.git
%cd Speech_Recognition

In [ ]:
!pip install -q -r requirements.txt

## Download LibriSpeech (train-clean-100)
Downloads ~6 GB from OpenSLR. Takes 10–20 minutes depending on connection.

In [ ]:
import os
import tarfile

DATA_DIR = 'data/raw/LibriSpeech/train-clean-100'
ARCHIVE = 'data/raw/train-clean-100.tar.gz'
URL = 'http://www.openslr.org/resources/12/train-clean-100.tar.gz'
MIN_SIZE = 6_000_000_000  # full file is ~6.3 GB

os.makedirs('data/raw', exist_ok=True)

def count_flacs():
    if not os.path.isdir(DATA_DIR):
        return 0
    return sum(1 for r, _, files in os.walk(DATA_DIR) for f in files if f.endswith('.flac'))

existing = count_flacs()
if existing >= 28000:
    print(f'Dataset already present ({existing} FLAC files). Skipping download.')
else:
    if os.path.exists(ARCHIVE) and os.path.getsize(ARCHIVE) < MIN_SIZE:
        bad_size = os.path.getsize(ARCHIVE)
        print(f'Removing broken partial download ({bad_size:,} bytes)...')
        os.remove(ARCHIVE)

    if not os.path.exists(ARCHIVE) or os.path.getsize(ARCHIVE) < MIN_SIZE:
        print('Downloading train-clean-100 (~6.3 GB). Expect 15-40 minutes...')
        print('Do not stop this cell until the download finishes.')
        !wget -c --tries=5 --timeout=60 --read-timeout=60 --show-progress -O {ARCHIVE} {URL}

    size = os.path.getsize(ARCHIVE)
    print(f'Archive size: {size / 1e9:.2f} GB')
    if size < MIN_SIZE:
        raise RuntimeError(
            'Download incomplete. Re-run this cell to resume (wget -c continues). '
            'If it keeps failing, download manually from https://www.openslr.org/12/ '
            'and upload to Colab Files, then extract with: '
            '!tar -xzf /content/train-clean-100.tar.gz -C data/raw/'
        )

    if not tarfile.is_tarfile(ARCHIVE):
        os.remove(ARCHIVE)
        raise RuntimeError('Downloaded file is not a valid archive (often an error page). Re-run to retry.')

    print('Extracting (5-10 minutes)...')
    !tar -xzf {ARCHIVE} -C data/raw/

    if count_flacs() < 28000:
        raise RuntimeError('Extraction finished but FLAC count looks wrong. Check data/raw/LibriSpeech/')

    print('Removing archive to free disk space...')
    !rm -f {ARCHIVE}
    print('Download and extract complete.')

flac_count = count_flacs()
print(f'FLAC files found: {flac_count}')
assert flac_count >= 28000, 'Expected ~28539 FLAC files'

## (Optional) Mount Google Drive to save the model
Recommended — Colab sessions disconnect and you lose files otherwise.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_MODEL_DIR = '/content/drive/MyDrive/Speech_Recognition/model'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
print('Model will be copied to:', DRIVE_MODEL_DIR)

## Train on full dataset
GPU: use batch size 32. Expect ~15–30 min per epoch.

In [ ]:
!python train.py --epochs 30 --batch-size 32 --num-workers 2

## Save model to Google Drive + download locally

In [ ]:
import shutil
from google.colab import files

if os.path.exists('model/lstm_ctc_model.pth'):
    if 'DRIVE_MODEL_DIR' in dir():
        shutil.copy('model/lstm_ctc_model.pth', f'{DRIVE_MODEL_DIR}/lstm_ctc_model.pth')
        shutil.copy('model/lstm_ctc_model_best.pth', f'{DRIVE_MODEL_DIR}/lstm_ctc_model_best.pth')
        print('Saved to Google Drive.')
    files.download('model/lstm_ctc_model.pth')
    print('Download started — place file in your local model/ folder for app.py')
else:
    print('Model not found. Training may not have finished.')